# Módulo 3 — Baseline supervisado con TF-IDF

**Objetivo:** establecer el punto de referencia clásico del proyecto — TF-IDF +
clasificador supervisado sobre el corpus preprocesado en el Módulo 2. Las métricas
de este notebook son la **columna A** de la tabla comparativa del Módulo 4.

**Protocolo anti-leakage:** la selección de configuración se hace contra el split de
validación (10% del train, estratificado); el test set se evalúa **una sola vez**, con
la configuración ya elegida. `fit_transform` siempre sobre train; `transform` sobre
validación/test.

In [ ]:
import sys
import time
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from src.config import SEED, PROCESSED_DIR, METRICS_DIR
from src.dataset import train_val_split
from src.evaluate import report, save_metrics, plot_confusion

train_full = pd.read_csv(PROCESSED_DIR / "train_clean.csv")
test = pd.read_csv(PROCESSED_DIR / "test_clean.csv")
tr, val = train_val_split(train_full, val_size=0.1)
print(f"train: {len(tr):,} | val: {len(val):,} | test: {len(test):,}")

## 1. Experimentación: tres configuraciones de TF-IDF

Se varían `max_features` y `ngram_range`; el resto de parámetros queda fijo
(`min_df=2`, `sublinear_tf=True`). El clasificador es Regresión Logística en las
tres corridas para aislar el efecto de la representación.

In [ ]:
CONFIGS = {
    "A": {"max_features": 20_000, "ngram_range": (1, 1)},
    "B": {"max_features": 50_000, "ngram_range": (1, 2)},
    "C": {"max_features": 100_000, "ngram_range": (1, 2)},
}

resultados = {}
for name, params in CONFIGS.items():
    tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True, **params)
    X_tr = tfidf.fit_transform(tr["text_clean"])
    X_val = tfidf.transform(val["text_clean"])
    clf = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=SEED)
    t0 = time.time()
    clf.fit(X_tr, tr["label"])
    t_fit = time.time() - t0
    f1_val = f1_score(val["label"], clf.predict(X_val), average="weighted")
    resultados[name] = {**params, "f1_val_weighted": round(f1_val, 4),
                        "fit_seconds": round(t_fit, 1)}
    print(f"config {name}: {params} -> F1 val {f1_val:.4f} ({t_fit:.0f}s)")

tabla = pd.DataFrame(resultados).T
tabla

## 2. Modelo final: mejor configuración, reentrenado sobre todo el train

**Justificación del clasificador:** Regresión Logística maneja bien matrices dispersas
de alta dimensión, entrena en segundos, expone probabilidades calibrables y sus
coeficientes son interpretables por clase — el baseline honesto contra el que medir
al Transformer. Naive Bayes es más rápido pero asume independencia entre features
(falsa con bi-gramas superpuestos); SVM lineal rinde parecido con más costo de ajuste.

In [ ]:
best_name = max(resultados, key=lambda k: resultados[k]["f1_val_weighted"])
best_params = CONFIGS[best_name]
print(f"mejor configuración en validación: {best_name} -> {best_params}")

tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True, **best_params)
X_train = tfidf.fit_transform(train_full["text_clean"])   # fit SOLO en train
X_test = tfidf.transform(test["text_clean"])

clf = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=SEED)
t0 = time.time()
clf.fit(X_train, train_full["label"])
train_seconds = time.time() - t0
print(f"entrenamiento final: {train_seconds:.0f}s | features: {X_train.shape[1]:,}")

## 3. Evaluación única sobre el test set

In [ ]:
y_pred = clf.predict(X_test)
rep = report(test["label"], y_pred)
plot_confusion(test["label"], y_pred, "m3_confusion",
               "Módulo 3 — matriz de confusión (TF-IDF + LogReg)")

n_params = clf.coef_.size + clf.intercept_.size
save_metrics(rep, "baseline_tfidf", extra={
    "modelo": "TfidfVectorizer + LogisticRegression",
    "configuracion": {**best_params, "min_df": 2, "sublinear_tf": True},
    "experimentos_validacion": resultados,
    "parametros_entrenables": n_params,
    "tiempo_entrenamiento_seg": round(train_seconds, 1),
})
print(f"parámetros del modelo: {n_params:,}")

## 4. Análisis: qué categorías cuestan más

<!-- ANALISIS_M3: completar tras la corrida con lectura de la matriz de confusión -->